# Fine-tune plate detector trên Colab (GPU miễn phí)

Trước khi chạy: vào **Runtime > Change runtime type > GPU** (T4 là đủ).

Cần chuẩn bị 1 file duy nhất từ máy bạn: `models/plate_best.pt` (model hiện tại, sẽ fine-tune tiếp từ đây, không train từ đầu). Dataset (8259 ảnh biển số VN) Colab tự tải, không cần bạn upload.

In [ ]:
!pip install -q ultralytics gdown

In [ ]:
# Upload plate_best.pt (nút Choose Files sẽ hiện ra, chọn file từ máy bạn)
from google.colab import files
uploaded = files.upload()  # chọn models/plate_best.pt

In [ ]:
# Tải dataset biển số VN thật (8259 ảnh, nguồn winter2897/Real-time-Auto-License-Plate-Recognition-with-Jetson-Nano)
!gdown "https://drive.google.com/uc?id=1KLK-DWgT3VoQH4fcTxAt2eB3sm7DGWAf" -O plate_yolo.zip
!unzip -q plate_yolo.zip -d raw

In [ ]:
# Tách train/val 90/10 — cùng logic đã chạy ở máy local, seed=42 để giống hệt
import os, random, shutil

src = "raw/yolo_plate_dataset"
dst = "vn_plate_detect"
random.seed(42)

imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.png'))
        and os.path.exists(os.path.join(src, os.path.splitext(f)[0] + '.txt'))]
random.shuffle(imgs)
n_val = max(1, int(len(imgs) * 0.1))
val_set = set(imgs[:n_val])

for split in ['train', 'val']:
    os.makedirs(f"{dst}/images/{split}", exist_ok=True)
    os.makedirs(f"{dst}/labels/{split}", exist_ok=True)

for f in imgs:
    split = 'val' if f in val_set else 'train'
    stem = os.path.splitext(f)[0]
    shutil.copy(os.path.join(src, f), f"{dst}/images/{split}/{f}")
    shutil.copy(os.path.join(src, stem + '.txt'), f"{dst}/labels/{split}/{stem}.txt")

print("total:", len(imgs), "train:", len(imgs) - n_val, "val:", n_val)

with open(f"{dst}/data.yaml", "w") as fp:
    fp.write(f"path: /content/{dst}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['plate']\n")

In [ ]:
from ultralytics import YOLO

model = YOLO("plate_best.pt")
model.train(
    data="vn_plate_detect/data.yaml",
    epochs=60,
    imgsz=640,
    batch=32,
    device=0,
    project="runs/train",
    name="plate_finetune_v1",
    patience=10,
)

In [ ]:
# Tải kết quả về máy — file này thay cho models/plate_best.pt, nhớ test lại trên ảnh thật
# (data/snapshots/) trước khi thay hẳn, đừng tin thẳng số liệu val của dataset.
from google.colab import files
files.download("runs/train/plate_finetune_v1/weights/best.pt")